In [1]:
import findspark
findspark.init("/opt/spark")

In [2]:
from pyspark.sql import SparkSession, functions as F

In [3]:
from pyspark.ml import Pipeline # pipeline to transform data
from pyspark.sql import SparkSession # to initiate spark
from pyspark.sql.types import FloatType
from pyspark.ml.feature import RegexTokenizer # tokenizer
from pyspark.ml.feature import HashingTF, IDF # vectorizer
from pyspark.ml.feature import StopWordsRemover # to remove stop words
from pyspark.sql.functions import concat_ws, col # to concatinate cols
from pyspark.ml.classification import LogisticRegression # ml model
from pyspark.ml.evaluation import MulticlassClassificationEvaluator # to evaluate the model

In [4]:
spark = SparkSession.builder.appName("News Text Classification").master("local[*]") \
.config("spark.driver.memory", "4g") \
.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/07/28 09:47:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/07/28 09:47:24 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [6]:
sc = spark.sparkContext

In [8]:
from pyspark import SparkFiles
p1 ="https://github.com/erkansirin78/datasets/raw/master/AG_News_Classification_Dataset/train.csv"
sc.addFile(p1)
df = spark.read.csv("file:///"+SparkFiles.get("*"), header=True, inferSchema=True).withColumnRenamed("Class Index", "label")

24/07/28 09:48:25 WARN SparkContext: The path https://github.com/erkansirin78/datasets/raw/master/AG_News_Classification_Dataset/train.csv has been added already. Overwriting of added paths is not supported in the current version.


In [9]:
df.cache()

DataFrame[label: int, Title: string, Description: string]

In [10]:
df.limit(5).toPandas()

,label,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


Class Index — Category of the news
1 — World
2 — Sports
3 — Business
4 — Science

## Concatenate textual columns to get a single text column

In [11]:
# Renaming 'Class Index' col to 'label'
df1 = df.withColumnRenamed('Class Index', 'label')

# Add a new column 'Text' by concatinating 'Title' and 'Description'
df2 = df1.withColumn("Text", concat_ws(" ", "Title", 'Description'))

# Remove old text columns
df3 = df2.select('label', 'Text')

In [12]:
df3.limit(5).toPandas()

,label,Text
0,3,Wall St. Bears Claw Back Into the Black (Reute...
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...
4,3,"Oil prices soar to all-time record, posing new..."


## Tokenizer — To convert the sentences to a list of words also known as tokens.

In [13]:
tokenizer = RegexTokenizer(inputCol="Text", outputCol="words", pattern="\\W")

## Stopwords remover — to remove stop words or meaningless words from the corpus (is, this, that, etc..)

In [14]:
stopwords_remover = StopWordsRemover(inputCol="words", outputCol="filtered")

## HashingTF — Calculating the Term Frequency of the words in the corpus

In [15]:
hashing_tf = HashingTF(inputCol="filtered",
                       outputCol="raw_features", 
                       numFeatures=10000)

## IDF Vectorizer —Inverse document frequency (IDF).

In [16]:
idf = IDF(inputCol="raw_features", outputCol="features")

## Train Test Split: keep 75% data as the train set and 25% data as a test set

In [17]:
(train, test) = df3.randomSplit([0.75, 0.25], seed = 142)

In [18]:
print("Training Dataset Count: " + str(train.count()))
print("Test Dataset Count: " + str(test.count()))

Training Dataset Count: 89972


Test Dataset Count: 30028


In [19]:
lr = LogisticRegression(featuresCol='features',
                        labelCol='label',
                        family="multinomial",
                        regParam=0.3,
                        elasticNetParam=0,
                        maxIter=50)

In [20]:
pipeline_object = Pipeline().setStages([tokenizer, stopwords_remover, hashing_tf, idf, lr])

In [21]:
pipeline_model = pipeline_object.fit(train)

24/07/28 09:49:46 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


In [22]:
transformed_df = pipeline_model.transform(test)

In [23]:
transformed_df.limit(10).toPandas()

,label,Text,words,filtered,raw_features,features,rawPrediction,probability,prediction
0,1,"#39;70,000 Darfur dead #39; since March About...","[39, 70, 000, darfur, dead, 39, since, march, ...","[39, 70, 000, darfur, dead, 39, since, march, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-8.0721971306452, 4.033926654418692, 1.433384...","[4.575356042294554e-06, 0.8280330137204931, 0....",1.0
1,1,"#39;A wonderful father, husband, brother and ...","[39, a, wonderful, father, husband, brother, a...","[39, wonderful, father, husband, brother, son,...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-8.072205185395896, 3.022323439431048, 2.1024...","[8.310657667324634e-06, 0.5469252251976755, 0....",1.0
2,1,#39;Batman #39; scales royal palace in protes...,"[39, batman, 39, scales, royal, palace, in, pr...","[39, batman, 39, scales, royal, palace, protes...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-8.072208540988735, 3.238020511419614, 2.0760...","[7.311248637760696e-06, 0.5969825269538668, 0....",1.0
3,1,#39;Black Widows #39; behind Beslan tragedy A...,"[39, black, widows, 39, behind, beslan, traged...","[39, black, widows, 39, behind, beslan, traged...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-8.072206938340793, 4.17139191218081, 1.09840...","[4.103055243907265e-06, 0.8519905618956471, 0....",1.0
4,1,#39;Deserter #39; surrenders at US base US Ar...,"[39, deserter, 39, surrenders, at, us, base, u...","[39, deserter, 39, surrenders, us, base, us, a...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-8.072196186782598, 3.6375013953317517, 1.730...","[6.061433224125837e-06, 0.7379592312616751, 0....",1.0
5,1,#39;MOMENT OF JOY #39; AS THUGS LET 2 WOMEN H...,"[39, moment, of, joy, 39, as, thugs, let, 2, w...","[39, moment, joy, 39, thugs, let, 2, women, ho...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-8.072198424796037, 3.1518556471639894, 2.467...","[7.361592668219786e-06, 0.5514631934308953, 0....",1.0
6,1,#39;Most people don #39;t know what to do - t...,"[39, most, people, don, 39, t, know, what, to,...","[39, people, 39, know, give, leave, 39, family...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-8.07218984951099, 3.0968380700883027, 2.5371...","[7.380728673713172e-06, 0.5232948179180313, 0....",1.0
7,1,#39;Passion #39; Source Named New Roman Catho...,"[39, passion, 39, source, named, new, roman, c...","[39, passion, 39, source, named, new, roman, c...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-8.072217008447545, 3.480381538442263, 2.1890...","[6.4912225563905236e-06, 0.6753926032884229, 0...",1.0
8,1,#39;Patience helped in freeing hostages #39; ...,"[39, patience, helped, in, freeing, hostages, ...","[39, patience, helped, freeing, hostages, 39, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-8.072199008505978, 3.2257978912108216, 1.817...","[7.618153570997636e-06, 0.6144795488875125, 0....",1.0
9,1,#39;Scorpion Queen #39; reclaims her crown Ku...,"[39, scorpion, queen, 39, reclaims, her, crown...","[39, scorpion, queen, 39, reclaims, crown, kua...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","(0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[-8.072205621217861, 2.441266354575973, 2.4519...","[9.483961833239503e-06, 0.3490861851869177, 0....",2.0


In [24]:
# to evalute model
evaluator = MulticlassClassificationEvaluator(predictionCol='prediction', metricName='accuracy')

# print test accuracy
print("Test-set Accuracy is : ", evaluator.evaluate(transformed_df))

Test-set Accuracy is :  0.9006260823231651


In [25]:
evaluator.getMetricName()

'accuracy'